# Verify Generated Output

This notebook validates the generated scenario data by comparing:
1. **Source**: Energy Agency Excel file (official annual totals)
2. **Generated**: Base scenario parquet files (hourly timeseries)

The hourly sums should exactly match the annual totals from the source.

## Setup

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

# Paths
GENERATOR_PATH = Path.cwd().parent
INPUT_PATH = GENERATOR_PATH / 'input'
OUTPUT_PATH = GENERATOR_PATH / 'output'

# Source files
EXCEL_PATH = INPUT_PATH / 'energy_agency_scenarios' / 'framtida-elbehov-pa-lansniva.xlsx'
DUCKDB_PATH = INPUT_PATH / 'energy_agency_scenarios' / 'energy_agency.duckdb'
BASE_PATH = OUTPUT_PATH / 'base'

print(f"Excel source: {EXCEL_PATH}")
print(f"  exists: {EXCEL_PATH.exists()}")
print(f"\nDuckDB source: {DUCKDB_PATH}")
print(f"  exists: {DUCKDB_PATH.exists()}")
print(f"\nBase output: {BASE_PATH}")
print(f"  exists: {BASE_PATH.exists()}")

## 1. Load Source Data (Energy Agency Excel)

In [ ]:
# Load raw Excel data
excel_df = pd.read_excel(EXCEL_PATH, sheet_name='Elbehov_Siffror', skiprows=1)
print(f"Loaded {len(excel_df)} rows from Excel")
print(f"\nColumns: {list(excel_df.columns)}")
excel_df.head()

In [ ]:
# Show source data totals by scenario and year
source_totals = excel_df.groupby(['Scenario', 'År'])['Elbehov (GWh)'].sum().reset_index()
source_totals.columns = ['scenario', 'year', 'total_gwh']

print("Source totals by scenario and year (GWh):")
pivot = source_totals.pivot(index='year', columns='scenario', values='total_gwh')
pivot

In [ ]:
# Show source data by sector (Level 1)
sector_totals = excel_df.groupby(['Scenario', 'Sektor nivå 1'])['Elbehov (GWh)'].sum().reset_index()
sector_totals.columns = ['scenario', 'sector', 'total_gwh']

print("Source totals by sector (all years combined, GWh):")
pivot_sector = sector_totals.pivot(index='sector', columns='scenario', values='total_gwh')
pivot_sector

## 2. Load Generated Data (Base Scenarios)

In [ ]:
# List available scenarios
scenarios = [p.name for p in BASE_PATH.iterdir() if p.is_dir()] if BASE_PATH.exists() else []
print(f"Found {len(scenarios)} base scenarios:")
for s in scenarios:
    parquet_path = BASE_PATH / s / 'data.parquet'
    size_mb = parquet_path.stat().st_size / 1024 / 1024 if parquet_path.exists() else 0
    print(f"  - {s} ({size_mb:.1f} MB)")

In [ ]:
# Load all scenarios using DuckDB for fast aggregation
con = duckdb.connect(':memory:')

if scenarios:
    con.execute(f"""
        CREATE VIEW hourly_demand AS
        SELECT * FROM read_parquet('{BASE_PATH}/*/data.parquet')
    """)
    
    # Show basic stats
    stats = con.execute("""
        SELECT 
            COUNT(*) as rows,
            COUNT(DISTINCT scenario_id) as scenarios,
            COUNT(DISTINCT segment) as segments,
            COUNT(DISTINCT geography) as geographies,
            MIN(timestamp) as min_ts,
            MAX(timestamp) as max_ts
        FROM hourly_demand
    """).fetchone()
    
    print(f"Generated data stats:")
    print(f"  Rows: {stats[0]:,}")
    print(f"  Scenarios: {stats[1]}")
    print(f"  Segments: {stats[2]}")
    print(f"  Geographies: {stats[3]}")
    print(f"  Period: {stats[4]} to {stats[5]}")
else:
    print("No scenarios found - run behovskartan2.ipynb first")

In [ ]:
# Show generated totals by scenario and year
if scenarios:
    generated_totals = con.execute("""
        SELECT 
            scenario_id as scenario,
            EXTRACT(YEAR FROM timestamp)::INT as year,
            SUM(value) as total_gwh
        FROM hourly_demand
        GROUP BY scenario_id, EXTRACT(YEAR FROM timestamp)::INT
        ORDER BY scenario, year
    """).fetchdf()
    
    print("Generated totals by scenario and year (GWh):")
    pivot_gen = generated_totals.pivot(index='year', columns='scenario', values='total_gwh')
    display(pivot_gen)

In [ ]:
# Show generated totals by segment
if scenarios:
    segment_totals_gen = con.execute("""
        SELECT 
            scenario_id as scenario,
            segment,
            SUM(value) as total_gwh
        FROM hourly_demand
        GROUP BY scenario_id, segment
        ORDER BY scenario, total_gwh DESC
    """).fetchdf()
    
    print("Generated totals by segment (all years combined, GWh):")
    pivot_seg = segment_totals_gen.pivot(index='segment', columns='scenario', values='total_gwh')
    display(pivot_seg)

## 3. Compare Source vs Generated

In [ ]:
# Load the DuckDB annual_demand table (processed source data)
if DUCKDB_PATH.exists():
    con.execute(f"ATTACH '{DUCKDB_PATH}' AS source (READ_ONLY)")
    
    # Get annual totals from source DuckDB
    source_annual = con.execute("""
        SELECT scenario, year, SUM(value) as source_gwh
        FROM source.annual_demand
        GROUP BY scenario, year
    """).fetchdf()
    
    print(f"Source annual demand: {len(source_annual)} rows")
    source_annual.head()

In [ ]:
# Compare source annual totals vs generated hourly sums
if scenarios and DUCKDB_PATH.exists():
    comparison = con.execute("""
        WITH generated AS (
            SELECT 
                scenario_id as scenario,
                EXTRACT(YEAR FROM timestamp)::INT as year,
                SUM(value) as generated_gwh
            FROM hourly_demand
            GROUP BY scenario_id, EXTRACT(YEAR FROM timestamp)::INT
        ),
        source AS (
            SELECT scenario, year, SUM(value) as source_gwh
            FROM source.annual_demand
            GROUP BY scenario, year
        )
        SELECT 
            s.scenario,
            s.year,
            s.source_gwh,
            g.generated_gwh,
            ABS(g.generated_gwh - s.source_gwh) as diff_gwh,
            ABS(g.generated_gwh - s.source_gwh) / s.source_gwh * 100 as diff_pct
        FROM source s
        JOIN generated g ON s.scenario = g.scenario AND s.year = g.year
        ORDER BY diff_pct DESC
    """).fetchdf()
    
    print("Comparison: Source vs Generated (top differences)")
    print(comparison.head(20).to_string(index=False))
    
    # Summary
    max_diff = comparison['diff_pct'].max()
    total_rows = len(comparison)
    exact_matches = (comparison['diff_gwh'] < 0.0001).sum()
    
    print(f"\nSummary:")
    print(f"  Total comparisons: {total_rows}")
    print(f"  Exact matches (< 0.0001 GWh): {exact_matches}")
    print(f"  Max difference: {max_diff:.6f}%")
    
    if exact_matches == total_rows:
        print("\n✓ All annual totals match exactly!")
    else:
        print(f"\n⚠ {total_rows - exact_matches} mismatches found")

## 4. Sample Hourly Data

In [ ]:
# Show a sample of hourly data for one scenario
if scenarios:
    sample = con.execute("""
        SELECT timestamp, segment, geography, value
        FROM hourly_demand
        WHERE scenario_id = (SELECT scenario_id FROM hourly_demand LIMIT 1)
        ORDER BY timestamp, segment, geography
        LIMIT 50
    """).fetchdf()
    
    print("Sample hourly data (first scenario):")
    sample

In [ ]:
# Show hourly pattern for one day
if scenarios:
    daily_pattern = con.execute("""
        SELECT 
            EXTRACT(HOUR FROM timestamp)::INT as hour,
            segment,
            SUM(value) as total_gwh
        FROM hourly_demand
        WHERE scenario_id = (SELECT scenario_id FROM hourly_demand LIMIT 1)
          AND timestamp >= '2030-01-15'
          AND timestamp < '2030-01-16'
        GROUP BY EXTRACT(HOUR FROM timestamp)::INT, segment
        ORDER BY segment, hour
    """).fetchdf()
    
    print("Hourly pattern for 2030-01-15:")
    pivot_daily = daily_pattern.pivot(index='hour', columns='segment', values='total_gwh')
    display(pivot_daily)

In [ ]:
# Cleanup
con.close()
print("Done.")